# Evo-Hex – Visualizações

Notebook para geração e customização dos gráficos do pipeline.  
Carrega os CSVs gerados pelas etapas 3-6 e salva os plots em `analysis/`.

**Pré-requisito:** execute as etapas do pipeline antes de rodar este notebook:
```bash
python main.py --steps 3 4 5 6
```

In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Garante que o pacote cath_analysis seja encontrado
ROOT = Path('/Volumes/promethion/cath')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ANALYSIS = ROOT / 'analysis'
ANALYSIS.mkdir(exist_ok=True)

from cath_analysis import plotting as P

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['savefig.dpi'] = 300

print(f'Pasta de saída: {ANALYSIS}')
print(f'CSVs disponíveis:')
for f in sorted(ANALYSIS.glob('*.csv')):
    print(f'  {f.name}')

## Carregar dados

In [ ]:
# ── Etapa 4: dados de hélices ─────────────────────────────────────────────────
def _load(name):
    p = ANALYSIS / name
    if p.exists():
        return pd.read_csv(p)
    print(f'  [faltando] {name} — execute a etapa correspondente do pipeline')
    return None

propensity_df  = _load('helix_propensities.csv')
positions_df   = _load('helix_positions.csv')

# ── Etapa 5: tipos de hélices ─────────────────────────────────────────────────
composition_df = _load('helix_type_composition.csv')
top_residues_df = _load('helix_type_top_residues.csv')
stat_df        = _load('helix_type_statistical_comparison.csv')

# ── Etapa 3: frequências ──────────────────────────────────────────────────────
freq_df        = _load('amino_acid_frequencies_per_structure.csv')

print('\nCarregamento concluído.')

---
## Grupo 1 – Análise básica de hélices

> Requer etapa 4 concluída.

In [ ]:
# Propensão observada vs Chou-Fasman
if propensity_df is not None:
    P.plot_helix_propensities(propensity_df, ANALYSIS)
    plt.show()

In [ ]:
# Preferência por posição na hélice (N-terminal / Meio / C-terminal)
if positions_df is not None:
    P.plot_helix_positions(positions_df, ANALYSIS)
    plt.show()

In [ ]:
# Distribuição de comprimentos
# helix_lengths não é salvo em CSV — carregue do cache do pipeline ou
# execute: from cath_analysis.helix_analysis import analyze_secondary_structure_with_dssp
print('helix_lengths requer re-execução da etapa 4 ou leitura do relatório TXT.')
print(f'Relatório disponível em: {ANALYSIS / "helix_analysis_comprehensive_report.txt"}')

---
## Grupo 2 – Tipos de hélices (H / G / I)

> Requer etapa 5 concluída.

In [ ]:
# Distribuição de tipos (bar + pie)
if composition_df is not None:
    counts = (
        composition_df.groupby('Helix_Type')['Count']
        .sum()
        .to_dict()
    )
    P.plot_helix_type_distribution(counts, ANALYSIS)
    plt.show()

In [ ]:
# Heatmap de composição por tipo (20 AAs × 3 tipos)
if composition_df is not None:
    P.plot_helix_type_composition(composition_df, ANALYSIS)
    plt.show()

In [ ]:
# Top 10 AAs por tipo
if top_residues_df is not None:
    P.plot_top_amino_acids_by_type(top_residues_df, ANALYSIS)
    plt.show()

In [ ]:
# Comparação estatística Alpha vs 3-10
if stat_df is not None:
    P.plot_statistical_differences(stat_df, ANALYSIS)
    plt.show()

---
## Grupos 3-6 – Análises evolutivas

> Requer etapa 6 concluída. Os dados evolutivos não são salvos em CSV automaticamente —
> execute o bloco abaixo para coletar (demora ~3-4h na primeira vez).

In [ ]:
# Coleta dados evolutivos (rode apenas se a etapa 6 ainda não foi executada)
# Descomente para rodar:

# from cath_analysis.evolutionary_analysis import (
#     collect_evolutionary_data,
#     compute_hydrophobic_moments,
#     compute_aa_cooccurrence,
#     compute_helix_length_composition,
#     compute_shannon_entropy_heptad,
#     compute_codon_degeneracy_vs_propensity,
#     compute_proteome_comparison,
# )
# from cath_analysis.config import STRUCTURES_CLEAN_PATH
# evo_data = collect_evolutionary_data(STRUCTURES_CLEAN_PATH)

print('Descomente as linhas acima para coletar dados evolutivos.')

In [ ]:
# Helical wheel composto
# if 'evo_data' in dir():
#     P.plot_helical_wheel_average(evo_data['helix_sequences'], ANALYSIS)
#     plt.show()

# Momento hidrofóbico de Eisenberg
# if 'evo_data' in dir():
#     moments = compute_hydrophobic_moments(evo_data['helix_sequences'])
#     P.plot_hydrophobic_moment_distribution(moments, ANALYSIS)
#     plt.show()

# N-cap / C-cap
# if 'evo_data' in dir():
#     P.plot_ncap_ccap(evo_data['ncap_position'], evo_data['ccap_position'], ANALYSIS)
#     plt.show()

print('Descomente os blocos acima após coletar evo_data.')

In [ ]:
# Distribuição de conteúdo helicoidal por proteína
# if 'evo_data' in dir():
#     P.plot_helix_content_distribution(evo_data['helix_content_per_structure'], ANALYSIS)
#     plt.show()

# Co-ocorrência de AAs em hélices
# if 'evo_data' in dir():
#     cooc_df = compute_aa_cooccurrence(evo_data['helix_sequences'])
#     P.plot_aa_cooccurrence(cooc_df, ANALYSIS)
#     plt.show()

# Comprimento vs composição
# if 'evo_data' in dir():
#     length_comp_df = compute_helix_length_composition(evo_data['per_helix_data'])
#     P.plot_helix_length_vs_composition(length_comp_df, ANALYSIS)
#     plt.show()

print('Descomente os blocos acima após coletar evo_data.')

In [ ]:
# Matriz de transição H→G→I
# if 'evo_data' in dir():
#     P.plot_helix_transition_matrix(evo_data['transitions'], ANALYSIS)
#     plt.show()

# Razão 3-10 por comprimento
# if 'evo_data' in dir():
#     P.plot_g_ratio_by_length(evo_data['helix_lengths_by_type'], ANALYSIS)
#     plt.show()

# Entropia de Shannon por posição do heptad
# if 'evo_data' in dir():
#     entropy_df = compute_shannon_entropy_heptad(evo_data['heptad_aa_distribution'])
#     P.plot_shannon_entropy_heptad(entropy_df, ANALYSIS)
#     plt.show()

print('Descomente os blocos acima após coletar evo_data.')

In [ ]:
# Degenerescência de códons vs propensão
# if propensity_df is not None:
#     codon_df = compute_codon_degeneracy_vs_propensity(propensity_df)
#     P.plot_codon_degeneracy_vs_propensity(codon_df, ANALYSIS)
#     plt.show()

# Enriquecimento vs proteoma humano
# if propensity_df is not None:
#     proteome_df = compute_proteome_comparison(propensity_df)
#     P.plot_proteome_comparison(proteome_df, ANALYSIS)
#     plt.show()

print('Descomente os blocos acima após coletar evo_data / propensity_df.')